In [1]:
import numpy as np
import tensorflow as tf
import unicodedata
from tensorflow.keras.layers import Input, LSTM, Dense
from tensorflow.keras.models import Model

def normalize(text):
    return unicodedata.normalize('NFC', text)

data = [
    ("hello", "नमस्ते"),
    ("how are you", "आप कैसे हैं"),
    ("i am fine", "मैं ठीक हूँ"),
    ("thank you", "धन्यवाद"),
    ("good night", "शुभ रात्रि")
]

inp_texts, tgt_texts = [], []
inp_chars, tgt_chars = set(), set()

for i,t in data:
  i=normalize(i)
  t=normalize(t)
  t='\t'+ t + '\n'

  inp_texts.append(i)
  tgt_texts.append(t)

  inp_chars.update(i)
  tgt_chars.update(t)

inp_chars = sorted(list(inp_chars))
tgt_chars = sorted(list(tgt_chars))

in_tok = {c:i for i,c in enumerate(inp_chars)}
tgt_tok = {c:i for i,c in enumerate(tgt_chars)}
rev_tg = {i:c for c,i in tgt_tok.items()}

max_in = max(len(x) for x in inp_texts)
max_tg = max(len(x) for x in tgt_texts)

enc_in = np.zeros((len(inp_texts),max_in,len(inp_chars)), dtype='float32')
dec_in = np.zeros((len(inp_texts),max_tg,len(tgt_chars)), dtype='float32')
dec_tar = np.zeros((len(inp_texts),max_tg,len(tgt_chars)), dtype='float32')

for i,(inp,tgt) in enumerate(zip(inp_texts,tgt_texts)):
  for t,c in enumerate(inp):
    enc_in[i,t,in_tok[c]] = 1
  for t,c in enumerate(tgt):
    dec_in[i,t,tgt_tok[c]] = 1
    if t>0:
      dec_tar[i,t-1,tgt_tok[c]] = 1

latent = 256
enc_inputs = Input(shape=(None,len(inp_chars)))
_, h, c = LSTM(latent,return_state=True)(enc_inputs)
enc_states = [h,c]

dec_inputs = Input(shape=(None,len(tgt_chars)))
dec_lstm = LSTM(latent,return_sequences=True,return_state=True)
dec_outputs, _, _ = dec_lstm(dec_inputs, initial_state=enc_states)
dense = Dense(len(tgt_chars),activation='softmax')
dec_out = dense(dec_outputs)

model = Model([enc_inputs,dec_inputs],dec_out)
model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])
model.fit([enc_in, dec_in], dec_tar, epochs=300, verbose=0)

enc_model = Model(enc_inputs, enc_states)

h_in = Input(shape=(latent,))
c_in = Input(shape=(latent,))


dec_out2, h, c = dec_lstm(dec_inputs, initial_state=[h_in, c_in])
dec_out2 = dense(dec_out2)

dec_model = Model(
    [dec_inputs, h_in, c_in],
    [dec_out2, h, c]
)

def decode(seq):
  states = enc_model.predict(seq, verbose=0)
  target = np.zeros((1,1,len(tgt_chars)))
  target[0,0,tgt_tok['\t']] = 1
  result = ""
  while True:
    pred, h, c = dec_model.predict([target] + states, verbose = 0)
    idx = np.argmax(pred[0, -1])
    char = rev_tg[idx]
    if char == '\n' or len(result)>max_tg:
      break

    result += char

    target = np.zeros((1,1,len(tgt_chars)))
    target[0,0,tgt_tok[char]] = 1
    states = [h,c]

  return result

for i in range(len(inp_texts)):
  print(inp_texts[i],"->", decode(enc_in[i:i+1]))

hello -> नमस्ते
how are you -> आप कैसे हैं
i am fine -> मैं ठीक हूँ
thank you -> धन्यवाद
good night -> शुभ रात्रि
